# MCE–ILCADT v5 — deteksi perubahan status terkonfirmasi (Colab)

Paket ini menjalankan SAP v5 (dibekukan 17 Oktober 2026). Model v5 membutuhkan hasil v4 analisis utama yang sudah ada di Drive (`MCE_FP_v4/runs/primary`) untuk pembanding "model v4 dipetakan".

**Aturan**
1. Runtime: **Runtime → Ubah jenis runtime → CPU, versi runtime 2026.04** (sama dengan v4 dan paper 1).
2. Jalankan sel 1, 2, 2b, 3 di awal setiap sesi, lalu satu sel sesi. Sel sesi bisa diulang bila runtime terputus.
3. Jangan mengubah file di `code/`, `data/`, atau folder `runs/` v4 dan v5.
4. Hasil uji tidak dicetak. Setelah semua sesi selesai, jalankan sel paket hasil dan unggah zip-nya ke chat.
5. Hasil data uji 2026 bersifat **eksploratif** (SAP v5 bagian 2).

Perkiraan waktu: ±30–40 menit per horizon.

In [ ]:
# 1. Mount Google Drive dan lokasi paket
from google.colab import drive
drive.mount('/content/drive')
PARENT = '/content/drive/MyDrive/Toddlers_Future_Prediction_BUTUR'   # sesuaikan bila nama folder berbeda
ROOT = f'{PARENT}/MCE_FP_v5'
V4_PRIMARY = f'{PARENT}/MCE_FP_v4/runs/primary'
import os; os.chdir(ROOT); print(sorted(os.listdir(ROOT))); print('v4 primary ada:', os.path.isdir(V4_PRIMARY))

In [ ]:
# 2. Pustaka — versi sama dengan paper 1 dan v4
!pip -q install tensorflow==2.20.0 scikit-learn==1.6.1 xgboost==3.1.2 lightgbm==4.6.0 catboost==1.2.8 pyarrow statsmodels patsy

In [ ]:
# 2b. Cek lingkungan = paper 1 / v4
import subprocess, sys, json, re
probe = r'''
import sys, json, os, re
import tensorflow as tf, sklearn, xgboost, lightgbm, catboost, numpy, pandas, statsmodels
cpu = re.search(r"model name\s*:\s*(.*)", open("/proc/cpuinfo").read())
print(json.dumps({"python": sys.version.split()[0], "tensorflow": tf.__version__, "scikit_learn": sklearn.__version__,
  "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__, "catboost": catboost.__version__,
  "numpy": numpy.__version__, "pandas": pandas.__version__, "statsmodels": statsmodels.__version__,
  "logical_cpus": os.cpu_count(), "cpu_model": cpu.group(1) if cpu else "unknown",
  "gpus": len(tf.config.list_physical_devices("GPU"))}))
'''
out = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
env = json.loads(out.stdout.strip().splitlines()[-1]); print(json.dumps(env, indent=1))
REQUIRED = {'tensorflow': '2.20.0', 'scikit_learn': '1.6.1', 'xgboost': '3.1.2', 'lightgbm': '4.6.0', 'catboost': '1.2.8'}
bad = {k: env[k] for k, v in REQUIRED.items() if env[k] != v}
assert not bad, f'Versi pustaka berbeda: {bad} — jalankan ulang sel 2.'
assert env['python'].startswith('3.12.'), 'Python bukan 3.12: pilih versi runtime 2026.04.'
assert env['gpus'] == 0, 'Runtime memakai GPU; pilih CPU.'
v4env = json.load(open(f'{V4_PRIMARY}/shared/environment_metadata_h1.json'))['versions']
same = {k: (v4env.get(k), env.get(k)) for k in ('python', 'numpy', 'pandas', 'scikit_learn', 'tensorflow')}
diff = {k: v for k, v in same.items() if v[0] != v[1]}
assert not diff, f'Lingkungan berbeda dari run v4 (pickle v4 mungkin tidak terbaca / hasil tidak sebanding): {diff}'
print('✓ Lingkungan sama dengan paper 1 dan run v4.')

In [ ]:
# 3. Verifikasi paket dan run v4 yang dibutuhkan
import hashlib, time, pathlib
EXPECTED = {
  'code/mce_ilcadt_fp_v5.py': '1010f55b4c3b1ebbf181bade1120988b22b118f684c2d9ff1551ed9da1731bed',
  'data/master_v4_pseudonymised.parquet': 'b6e73b23911ddd02c9d04b5b390fa0745d9a41c1be55efd676496644815b2a1a',
  'SAP_v5.md': 'dab1e43e42c6e696a963fbba845ab6bbee9ae0096a7ce618993fcb04c48a1aa0',
}
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
for p, h in EXPECTED.items():
    assert sha(f'{ROOT}/{p}') == h, f'BERBEDA: {p}'
for h in (1, 3, 6):
    art = f'{V4_PRIMARY}/h{h}/artifacts'
    for f in ('freeze_manifest.json', 'stage_f_manifest.json', 'tune_pairs.pkl', 'test_pairs.pkl',
              'meta_dnn_tune_probabilities.npy', 'meta_dnn_test_probabilities.npy'):
        assert os.path.exists(f'{art}/{f}'), f'v4 h={h}: {f} tidak ditemukan'
print('✓ Kode, data, SAP v5, dan artefak v4 (h = 1, 3, 6) lengkap.')

RUNS = f'{ROOT}/runs'
CONFIGS = {'primary': {}, 'sens_no_baseline_confirmation': {'MCE_FP_V5_TARGET': 'no_baseline_confirmation'},
           'sens_unconfirmed': {'MCE_FP_V5_TARGET': 'unconfirmed'}, 'rerun_primary': {}}

def _call(config, stages, horizons):
    env = {**os.environ, **CONFIGS[config],
           'MCE_FP_MASTER_PATH': f'{ROOT}/data/master_v4_pseudonymised.parquet', 'MCE_FP_SAP_PATH': f'{ROOT}/SAP_v5.md',
           'MCE_FP_V4_RUN_ROOT': V4_PRIMARY, 'MCE_FP_BASE_DIR': f'{RUNS}/{config}',
           'MCE_FP_STAGES': stages, 'MCE_FP_HORIZONS': horizons}
    log = pathlib.Path(f'{RUNS}/{config}/logs'); log.mkdir(parents=True, exist_ok=True)
    logfile = log / f"{time.strftime('%Y%m%d_%H%M%S')}_{stages.replace(',', '')}_h{horizons.replace(',', '-')}.log"
    t = time.time()
    with open(logfile, 'w') as f:
        r = subprocess.run([sys.executable, f'{ROOT}/code/mce_ilcadt_fp_v5.py'], env=env, stdout=f, stderr=subprocess.STDOUT)
    lines = [l for l in open(logfile) if l.startswith(('✓', '  h=', 'Traceback', 'RuntimeError', 'ValueError'))]
    print(''.join(lines[-12:]) + f'   ({(time.time() - t) / 60:.1f} menit; log: {logfile.name})')
    if r.returncode != 0:
        raise RuntimeError(f'Gagal — lihat {logfile}')

def status(config, h):
    art = pathlib.Path(f'{RUNS}/{config}/h{h}/artifacts')
    if (art / 'stage_f_manifest.json').exists(): return 'selesai'
    if (art / 'freeze_manifest.json').exists(): return 'dibekukan'
    if (art / 'stage_b_manifest.json').exists(): return 'pasangan siap'
    return 'belum'

def run(config, horizons=(1, 3, 6)):
    if any(status(config, h) == 'belum' for h in horizons):
        _call(config, 'a,b', ','.join(map(str, horizons)))
    for h in horizons:
        s = status(config, h)
        if s == 'selesai':
            print(f'{config} h={h}: sudah selesai, dilewati'); continue
        stages = 'a,f' if s == 'dibekukan' else 'a,c,d,e,f'
        print(f'{config} h={h}: menjalankan tahap {stages} …')
        _call(config, stages, str(h))
print('Fungsi siap.')

### S1 — Analisis utama (target terkonfirmasi): h = 1, 3, 6

In [ ]:
run('primary')

### S2 — Sensitivitas 1: tanpa konfirmasi bulan t

In [ ]:
run('sens_no_baseline_confirmation')

### S3 — Sensitivitas 2: tanpa konfirmasi sama sekali

In [ ]:
run('sens_unconfirmed')

### Status dan FREEZE_LOG_v5

In [ ]:
import pandas as pd
display(pd.DataFrame([{'analisis': c, 'h': h, 'status': status(c, h)} for c in CONFIGS for h in (1, 3, 6)]))
for c in CONFIGS:
    p = f'{RUNS}/{c}/shared/FREEZE_LOG_v5.txt'
    if os.path.exists(p): print(open(p).read())

### S4 — Run ulang independen analisis utama (runtime baru, setelah S1)
Putuskan dan hapus runtime dulu, lalu sel 1, 2, 2b, 3, kemudian sel ini.

In [ ]:
run('rerun_primary')
import numpy as np
for h in (1, 3, 6):
    a, b = f'{RUNS}/primary/h{h}/artifacts', f'{RUNS}/rerun_primary/h{h}/artifacts'
    same = {f: bool(np.array_equal(np.load(f'{a}/{f}'), np.load(f'{b}/{f}')))
            for f in ('oof_meta_dev.npy', 'meta_dnn_test_probabilities.npy', 'test_decisions.npy')}
    same['comparators'] = all(np.array_equal(np.load(f'{a}/comparator_test_probabilities.npz')[k], np.load(f'{b}/comparator_test_probabilities.npz')[k])
                              for k in np.load(f'{a}/comparator_test_probabilities.npz').files)
    same['primary_metrics'] = open(f'{a}/primary_endpoint_metrics.csv').read() == open(f'{b}/primary_endpoint_metrics.csv').read()
    print(f'h={h}', same)

### Paket hasil untuk diunggah ke chat (CSV/JSON/log saja)

In [ ]:
import zipfile
out = f'{ROOT}/MCE_FP_v5_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path(RUNS).rglob('*'):
        if p.is_file() and p.suffix in ('.csv', '.json', '.txt', '.log') and 'base_fold_bundles' not in p.parts:
            z.write(p, p.relative_to(ROOT))
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')